# Notebook 5: The Regularization Path

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 2 hours  
**Topic:** Regularization & Optimization — Week 8  
**Dataset:** Synthetic California-style house price data

---

## Why This Matters

Picking a single regularization strength α is just one decision. But what if you could watch *how* your model changes across **every possible α** at once? The regularization path gives you exactly that — a movie instead of a snapshot. It reveals:

- Which features the model trusts most (they survive the longest as α grows)
- How aggressively each feature's coefficient shrinks
- The structural difference between Lasso (sparse) and Ridge (dense) shrinkage
- Where the optimal α lives relative to feature entry/exit points

Understanding the path turns hyperparameter tuning from a black box into an informed diagnostic tool.

## Real-World Analogy: The Real Estate Portfolio Budget

Imagine you manage a real estate portfolio worth **\$10 million** spread across **20 properties**. Suddenly you must shrink your budget:

| Budget Remaining | Properties You Keep |
|------------------|---------------------|
| \$10M (full)     | All 20 properties   |
| \$5M (half)      | Top 12 properties   |
| \$2M (tight)     | Top 5 properties    |
| \$500K (severe)  | Top 2 properties    |

As your budget (= allowed coefficient magnitude) shrinks, you **sell the least valuable properties first**. The last property you'd ever sell is your most prized asset.

**That is the regularization path.** The x-axis is your shrinking budget (increasing α). Each line is one property (feature). The last feature to reach zero is your single most important predictor.

> **Key insight:** The *order* in which features drop to zero tells you their relative importance — no extra computation needed.

## Plain English: What Is a Regularization Path?

A **regularization path** is the sequence of coefficient vectors β(α) you get by solving the penalized regression at many values of α:

- **Small α** → almost no penalty → coefficients near OLS values (large, possibly overfitting)
- **Large α** → heavy penalty → coefficients shrink toward zero (underfitting)

By sweeping α from near-zero to very large, you trace out a *path* through coefficient space.

### Lasso Path (L1 penalty)
$$\hat{\beta}^{\text{Lasso}}(\alpha) = \arg\min_{\beta} \left[ \frac{1}{2n}\|y - X\beta\|_2^2 + \alpha \|\beta\|_1 \right]$$

Lasso paths have **kinks**: each feature hits exactly zero at some threshold α. The geometry of the L1 ball (a diamond in 2D) creates corners that attract solutions.

### Ridge Path (L2 penalty)
$$\hat{\beta}^{\text{Ridge}}(\alpha) = \arg\min_{\beta} \left[ \frac{1}{2n}\|y - X\beta\|_2^2 + \alpha \|\beta\|_2^2 \right]$$

Ridge paths are **smooth curves**: all coefficients shrink asymptotically toward zero but *never reach it*. The L2 ball is a sphere — no corners, no exact zeros.

### ElasticNet Path (mix of L1 + L2)
$$\hat{\beta}^{\text{EN}}(\alpha) = \arg\min_{\beta} \left[ \frac{1}{2n}\|y - X\beta\|_2^2 + \alpha \left( \rho\|\beta\|_1 + \frac{1-\rho}{2}\|\beta\|_2^2 \right) \right]$$

ElasticNet paths show kinks (from L1) but softer shrinkage than pure Lasso (from L2).

## Section 1: Setup & Synthetic House Price Dataset

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

from sklearn.linear_model import Lasso, Ridge, ElasticNet, lasso_path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline

# Reproducibility & aesthetics
np.random.seed(42)
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

print('Libraries loaded. NumPy', np.__version__, '| scikit-learn imported')

In [ ]:
# ── Build synthetic California-style house price dataset ─────────────────────
# 300 samples, 15 features:
#   8 real features with non-zero true coefficients
#   7 noise features (true coefficient = 0)

n_samples   = 300
n_real      = 8      # informative features
n_noise     = 7      # pure noise features
n_features  = n_real + n_noise

# Feature names
real_names  = ['sq_ft', 'bedrooms', 'bathrooms', 'lot_size',
               'age_years', 'garage', 'school_rating', 'distance_cbd']
noise_names = [f'noise_{i}' for i in range(1, n_noise + 1)]
feat_names  = real_names + noise_names

# Generate correlated feature matrix
X_raw = np.random.randn(n_samples, n_features)

# True coefficients: real features have known weights, noise = 0
true_coef = np.array([150, 20, 35, 40, -8, 25, 18, -30,   # real
                       0,  0,  0,  0,  0,  0,  0])         # noise

noise_std = 50  # house price noise in $K
y = X_raw @ true_coef + 250 + np.random.randn(n_samples) * noise_std

# Wrap in a DataFrame for readability
df = pd.DataFrame(X_raw, columns=feat_names)
df['price_kUSD'] = y

print(f'Dataset shape: {df.shape}')
print(f'Price range: ${y.min():.0f}K – ${y.max():.0f}K')
print(f'\nFirst 3 rows:')
df.head(3)

In [ ]:
# ── Standardize features (critical before regularization) ────────────────────
# Regularization penalizes coefficient magnitude.
# If features are on different scales, the penalty is unfair:
# a feature in meters vs. one in mm would be penalized unequally.

X = df[feat_names].values
y_vec = df['price_kUSD'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Color palette: real features = blue family, noise = red/orange family
real_colors  = sns.color_palette('Blues_d',  n_real)[::-1]
noise_colors = sns.color_palette('Oranges_d', n_noise)[::-1]
all_colors   = real_colors + noise_colors

print('Features standardized (mean=0, std=1).')
print(f'Real feature colors: blue shades ({n_real} features)')
print(f'Noise feature colors: orange shades ({n_noise} features)')

## Section 2: Computing the Lasso Regularization Path from Scratch

We loop over 100 logarithmically-spaced α values, fit Lasso at each, and record the resulting coefficient vector. This is the brute-force approach — slow but transparent.

In [ ]:
# ── Lasso path: manual loop over alpha values ─────────────────────────────────
# We use 'warm_start': each model initializes from the previous alpha's solution.
# This dramatically speeds up convergence along the path.

alphas_manual = np.logspace(-3, 2, 100)  # 0.001 → 100, 100 values
coefs_manual  = np.zeros((len(alphas_manual), n_features))

lasso_warm = Lasso(alpha=alphas_manual[0], max_iter=10000, warm_start=True)

for i, a in enumerate(alphas_manual):
    lasso_warm.set_params(alpha=a)
    lasso_warm.fit(X_scaled, y_vec)
    coefs_manual[i, :] = lasso_warm.coef_

# How many features survive (non-zero) at each alpha?
n_nonzero = (coefs_manual != 0).sum(axis=1)

print(f'Alpha range: {alphas_manual[0]:.4f} → {alphas_manual[-1]:.1f}')
print(f'At alpha={alphas_manual[0]:.4f}: {n_nonzero[0]} non-zero coefficients')
print(f'At alpha={alphas_manual[50]:.3f}: {n_nonzero[50]} non-zero coefficients')
print(f'At alpha={alphas_manual[-1]:.1f}: {n_nonzero[-1]} non-zero coefficients')

## Section 3: Plotting the Lasso Regularization Path

In [ ]:
# ── Find the cross-validated optimal alpha for Lasso ─────────────────────────
# We'll use this to draw a vertical reference line on the path plot.

from sklearn.linear_model import LassoCV

lasso_cv = LassoCV(alphas=alphas_manual, cv=5, max_iter=10000, random_state=42)
lasso_cv.fit(X_scaled, y_vec)
optimal_alpha = lasso_cv.alpha_

print(f'Cross-validated optimal alpha: {optimal_alpha:.4f}')
print(f'Number of non-zero features at optimal alpha: '
      f'{(lasso_cv.coef_ != 0).sum()}')

In [ ]:
# ── Plot the Lasso regularization path ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ─ Left panel: all features, full path ─
ax = axes[0]
for j, name in enumerate(feat_names):
    lw    = 2.5 if j < n_real else 1.2
    ls    = '-'  if j < n_real else '--'
    label = name if j < n_real else (name if j == n_real else None)  # one noise label
    ax.plot(np.log10(alphas_manual), coefs_manual[:, j],
            color=all_colors[j], lw=lw, ls=ls)
    # Annotate real features at the left edge
    if j < n_real and abs(coefs_manual[0, j]) > 5:
        ax.text(np.log10(alphas_manual[0]) - 0.05, coefs_manual[0, j],
                name, fontsize=8, va='center', ha='right', color=all_colors[j])

ax.axvline(np.log10(optimal_alpha), color='red', lw=2, ls=':', label=f'Optimal α={optimal_alpha:.3f}')
ax.axhline(0, color='black', lw=0.7)
ax.set_xlabel('log₁₀(α)  →  increasing regularization')
ax.set_ylabel('Coefficient value')
ax.set_title('Lasso Regularization Path\n(blue = real features, orange = noise features)')
ax.legend(loc='upper right', fontsize=9)

# ─ Right panel: number of non-zero features vs alpha ─
ax2 = axes[1]
ax2.plot(np.log10(alphas_manual), n_nonzero, color='steelblue', lw=2.5)
ax2.axvline(np.log10(optimal_alpha), color='red', lw=2, ls=':',
            label=f'Optimal α={optimal_alpha:.3f}')
ax2.axhline(n_real, color='green', lw=1.5, ls='--', label=f'{n_real} real features')
ax2.set_xlabel('log₁₀(α)  →  increasing regularization')
ax2.set_ylabel('# Non-zero coefficients')
ax2.set_title('Sparsity vs. Regularization Strength')
ax2.legend(fontsize=9)
ax2.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.suptitle('Figure 1: Lasso Regularization Path — House Price Prediction', y=1.02, fontsize=13)
plt.show()

print('Observation: noise features (orange dashed) hit zero at smaller alpha values')
print('than real features (blue solid). The most informative features survive longest.')

## Section 4: sklearn's `lasso_path` — Efficient Analytical Solution

sklearn's `lasso_path` uses the **LARS algorithm** (Least Angle Regression) to compute the *exact* Lasso path analytically — no loops, no warm starting needed. It's much faster and numerically more precise.

In [ ]:
# ── sklearn lasso_path: analytical computation ────────────────────────────────
# lasso_path returns:
#   alphas_sk: the alpha values it chose (or the ones we specify)
#   coefs_sk:  shape (n_features, n_alphas) — note: transposed vs. our manual array!

alphas_sk, coefs_sk, _ = lasso_path(
    X_scaled, y_vec,
    alphas=alphas_manual,    # use the same alpha grid for comparison
    max_iter=10000
)

# Transpose so shape is (n_alphas, n_features) — same as coefs_manual
coefs_sk_T = coefs_sk.T

# ─ Verify agreement between manual loop and lasso_path ─
max_diff = np.abs(coefs_manual - coefs_sk_T).max()
print(f'Maximum coefficient difference (manual vs lasso_path): {max_diff:.6f}')
print(f'(Values near 0 confirm both methods produce the same path)')

# ─ Compare coefficient at optimal alpha ─
opt_idx = np.argmin(np.abs(alphas_manual - optimal_alpha))
print(f'\nCoefficients at optimal alpha ({optimal_alpha:.4f}):')
coef_df = pd.DataFrame({
    'feature':       feat_names,
    'true_coef':     true_coef,
    'manual_loop':   coefs_manual[opt_idx],
    'lasso_path':    coefs_sk_T[opt_idx]
})
coef_df['is_real'] = coef_df['feature'].isin(real_names)
print(coef_df.sort_values('manual_loop', ascending=False).to_string(index=False))

## Section 5: Ridge Regularization Path — Smooth Shrinkage Without Zeros

In [ ]:
# ── Compute Ridge path ────────────────────────────────────────────────────────
# Ridge has a closed-form solution: beta = (X'X + alpha*I)^{-1} X'y
# So there are no convergence issues; we can sweep alpha freely.

alphas_ridge = np.logspace(-3, 4, 150)   # wider range for Ridge
coefs_ridge  = np.zeros((len(alphas_ridge), n_features))

for i, a in enumerate(alphas_ridge):
    ridge = Ridge(alpha=a, fit_intercept=True)
    ridge.fit(X_scaled, y_vec)
    coefs_ridge[i, :] = ridge.coef_

# ─ Plot ─
fig, ax = plt.subplots(figsize=(12, 6))
for j, name in enumerate(feat_names):
    lw = 2.5 if j < n_real else 1.2
    ls = '-'  if j < n_real else '--'
    ax.plot(np.log10(alphas_ridge), coefs_ridge[:, j],
            color=all_colors[j], lw=lw, ls=ls)
    if j < n_real and abs(coefs_ridge[0, j]) > 5:
        ax.text(np.log10(alphas_ridge[0]) - 0.1, coefs_ridge[0, j],
                name, fontsize=8, va='center', ha='right', color=all_colors[j])

ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('log₁₀(α)  →  increasing regularization')
ax.set_ylabel('Coefficient value')
ax.set_title('Ridge Regularization Path — All Coefficients Shrink Smoothly\n'
             '(Note: no coefficient ever reaches exactly zero)')

# Annotate the smooth-shrinkage property
ax.annotate('Ridge coefficients approach\n0 asymptotically — never reach it',
            xy=(3.5, 5), xytext=(2.0, 40),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=10, color='darkred')

plt.tight_layout()
plt.suptitle('Figure 2: Ridge Regularization Path — House Price Prediction', y=1.01, fontsize=13)
plt.show()

print('Key difference from Lasso: Ridge shrinks ALL coefficients toward zero,')
print('but never achieves exact sparsity. Feature selection is NOT possible with Ridge alone.')

## Section 6: ElasticNet Path — The Hybrid (l1_ratio = 0.5)

ElasticNet combines L1 (Lasso) and L2 (Ridge) penalties. With `l1_ratio=0.5`, it's an equal mix: you get *some* sparsity (kinks) but with softer, more stable shrinkage than pure Lasso.

In [ ]:
# ── Compute ElasticNet path at l1_ratio=0.5 ───────────────────────────────────
from sklearn.linear_model import enet_path

l1_ratio = 0.5
alphas_en = np.logspace(-3, 2, 100)

_, coefs_en, _ = enet_path(
    X_scaled, y_vec,
    l1_ratio=l1_ratio,
    alphas=alphas_en,
    max_iter=10000
)
coefs_en = coefs_en.T   # shape: (n_alphas, n_features)

# ─ Three-panel comparison: Lasso vs Ridge vs ElasticNet ─
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

path_data = [
    (alphas_manual, coefs_manual, 'Lasso (L1, pure sparse)'),
    (alphas_ridge[:100], coefs_ridge[:100],  'Ridge (L2, smooth)'),
    (alphas_en,    coefs_en,    f'ElasticNet (l1_ratio={l1_ratio})')
]

for ax, (alphas_p, coefs_p, title) in zip(axes, path_data):
    for j, name in enumerate(feat_names):
        lw = 2.5 if j < n_real else 1.0
        ls = '-'  if j < n_real else '--'
        ax.plot(np.log10(alphas_p), coefs_p[:, j],
                color=all_colors[j], lw=lw, ls=ls)
    ax.axhline(0, color='black', lw=0.7)
    ax.set_xlabel('log₁₀(α)')
    ax.set_ylabel('Coefficient')
    ax.set_title(title, fontsize=10)

# Add a shared legend patch
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='steelblue', lw=2.5, label='Real feature'),
    Line2D([0], [0], color='orange',    lw=1.0, ls='--', label='Noise feature')
]
fig.legend(handles=legend_elements, loc='lower center', ncol=2,
           fontsize=10, bbox_to_anchor=(0.5, -0.08))

plt.suptitle('Figure 3: Regularization Paths Compared\nLasso | Ridge | ElasticNet', fontsize=13)
plt.tight_layout()
plt.show()

print('Lasso: hard kinks, exact zeros.  Ridge: smooth curves, no zeros.')
print('ElasticNet: kinks present but softer than Lasso — correlated features handled better.')

## Section 7: Feature Importance Ranking from the Lasso Path

The order in which features **become active** (as α decreases from large to small) directly reflects importance. The first feature to "enter" is the most important single predictor.

In [ ]:
# ── Feature importance: alpha at which each feature first becomes non-zero ────
# We sweep alpha from LARGE to SMALL (right to left on the path plot).
# The first alpha where |coef| > threshold = "entry alpha".
# Features with large entry alpha are the most important.

threshold = 1e-4  # treat coefficients below this as zero

entry_alpha = {}
# coefs_manual is sorted small→large alpha; reverse to find entry from large side
coefs_rev = coefs_manual[::-1]          # now index 0 = largest alpha
alphas_rev = alphas_manual[::-1]

for j, name in enumerate(feat_names):
    # Find first row (highest alpha) where |coef| > threshold
    active = np.where(np.abs(coefs_rev[:, j]) > threshold)[0]
    if len(active) > 0:
        entry_alpha[name] = alphas_rev[active[0]]
    else:
        entry_alpha[name] = 0.0  # never becomes active

# Build importance DataFrame
importance_df = pd.DataFrame({
    'feature':     list(entry_alpha.keys()),
    'entry_alpha': list(entry_alpha.values()),
    'is_real':     [name in real_names for name in entry_alpha.keys()],
    'true_coef':   [true_coef[feat_names.index(n)] for n in entry_alpha.keys()]
}).sort_values('entry_alpha', ascending=False)

print('Feature importance ranking (by Lasso entry alpha):')
print('Higher entry_alpha = more important (entered path at stronger regularization)')
print(importance_df.to_string(index=False))

In [ ]:
# ── Bar chart: feature importance ranking ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

colors_bar = ['steelblue' if r else 'darkorange' for r in importance_df['is_real']]
bars = ax.barh(importance_df['feature'], importance_df['entry_alpha'],
               color=colors_bar, edgecolor='black', linewidth=0.6)

# Annotate with true coefficient magnitude
for bar, tc in zip(bars, importance_df['true_coef']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
            f'β={tc:.0f}', va='center', ha='left', fontsize=8)

ax.set_xlabel('Entry alpha (higher = more important)')
ax.set_title('Feature Importance from Lasso Regularization Path\n'
             '(blue = real features, orange = noise features)')
ax.axvline(0, color='black', lw=0.8)

from matplotlib.patches import Patch
legend_patches = [
    Patch(color='steelblue', label='Real feature (true β ≠ 0)'),
    Patch(color='darkorange', label='Noise feature (true β = 0)')
]
ax.legend(handles=legend_patches, fontsize=9)
plt.tight_layout()
plt.show()

print('\nDid Lasso correctly identify real vs. noise features?')
top_8 = importance_df.head(8)['feature'].tolist()
correct = sum(1 for f in top_8 if f in real_names)
print(f'Top 8 features by entry alpha: {correct}/8 are actual real features')

## Section 8: Cross-Validation for Optimal α — Connecting Path to Model Selection

In [ ]:
# ── CV error vs. alpha: find optimal alpha and mark it on the path ─────────────
# We compute 5-fold CV MSE for each alpha in our grid.

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_mean = np.zeros(len(alphas_manual))
cv_scores_std  = np.zeros(len(alphas_manual))

for i, a in enumerate(alphas_manual):
    lasso_cv_model = Lasso(alpha=a, max_iter=10000)
    scores = cross_val_score(lasso_cv_model, X_scaled, y_vec,
                             cv=kf, scoring='neg_mean_squared_error')
    cv_scores_mean[i] = -scores.mean()   # convert to positive MSE
    cv_scores_std[i]  = scores.std()

cv_optimal_idx   = np.argmin(cv_scores_mean)
cv_optimal_alpha = alphas_manual[cv_optimal_idx]

print(f'CV-optimal alpha: {cv_optimal_alpha:.5f}')
print(f'CV MSE at optimal: {cv_scores_mean[cv_optimal_idx]:.2f}')
print(f'Number of active features at CV-optimal alpha: '
      f'{(coefs_manual[cv_optimal_idx] != 0).sum()}')

In [ ]:
# ── Combined figure: CV curve + path with optimal alpha marked ────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 10), sharex=True)

log_alphas = np.log10(alphas_manual)

# ─ Top: CV error curve ─
ax1.plot(log_alphas, cv_scores_mean, color='navy', lw=2.5, label='CV MSE (mean)')
ax1.fill_between(log_alphas,
                 cv_scores_mean - cv_scores_std,
                 cv_scores_mean + cv_scores_std,
                 alpha=0.25, color='navy', label='±1 std')
ax1.axvline(np.log10(cv_optimal_alpha), color='red', lw=2, ls=':',
            label=f'Optimal α={cv_optimal_alpha:.4f}')
ax1.set_ylabel('Cross-Validation MSE')
ax1.set_title('Top: CV Error vs. Regularization Strength\n'
              'Bottom: Lasso Path (feature coefficients)')
ax1.legend(fontsize=9)

# ─ Bottom: Lasso path ─
for j, name in enumerate(feat_names):
    lw = 2.5 if j < n_real else 1.0
    ls = '-'  if j < n_real else '--'
    ax2.plot(log_alphas, coefs_manual[:, j], color=all_colors[j], lw=lw, ls=ls)
    if j < n_real and abs(coefs_manual[0, j]) > 5:
        ax2.text(log_alphas[0] - 0.05, coefs_manual[0, j],
                 name, fontsize=8, va='center', ha='right', color=all_colors[j])

ax2.axvline(np.log10(cv_optimal_alpha), color='red', lw=2, ls=':',
            label=f'Optimal α={cv_optimal_alpha:.4f}')
ax2.axhline(0, color='black', lw=0.7)
ax2.set_xlabel('log₁₀(α)  →  increasing regularization')
ax2.set_ylabel('Coefficient value')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.suptitle('Figure 4: Connecting CV Model Selection to the Regularization Path',
             y=1.01, fontsize=13)
plt.show()

print('The red dashed line shows where CV says to stop regularizing.')
print('Features still active at that line are the ones CV considers useful.')

## Section 9: Path Summary — What Did We Learn?

In [ ]:
# ── Summary table of coefficient behavior ─────────────────────────────────────
opt_coefs_lasso = coefs_manual[cv_optimal_idx]

# Ridge at same alpha for comparison
ridge_opt = Ridge(alpha=cv_optimal_alpha).fit(X_scaled, y_vec)
opt_coefs_ridge = ridge_opt.coef_

# ElasticNet at same alpha
en_opt = ElasticNet(alpha=cv_optimal_alpha, l1_ratio=0.5, max_iter=10000).fit(X_scaled, y_vec)
opt_coefs_en = en_opt.coef_

summary_df = pd.DataFrame({
    'feature':         feat_names,
    'true_coef':       true_coef,
    'lasso_coef':      opt_coefs_lasso,
    'ridge_coef':      opt_coefs_ridge,
    'elasticnet_coef': opt_coefs_en,
    'is_real':         [name in real_names for name in feat_names]
})

# Mark zero vs. non-zero for Lasso
summary_df['lasso_active'] = summary_df['lasso_coef'].abs() > 1e-4

print(f'Summary at optimal alpha = {cv_optimal_alpha:.4f}:')
print(summary_df.round(3).to_string(index=False))

n_correct = ((summary_df['is_real']) & (summary_df['lasso_active'])).sum()
n_false   = ((~summary_df['is_real']) & (summary_df['lasso_active'])).sum()
print(f'\nLasso correctly activates {n_correct}/{n_real} real features')
print(f'Lasso incorrectly keeps {n_false} noise features')

## Self-Check Questions

Test your understanding before moving on.

---

**Question 1:** Looking at the Lasso regularization path, Feature A enters first (at large α). Feature B enters last. Which feature is more important and why?

<details>
<summary>Show Answer</summary>

**Feature A is more important.** The Lasso path is read from right (large α = heavy penalty) to left (small α = light penalty). A feature that "enters" at a *large* α means it is strong enough to overcome the heavy penalty and still get a non-zero coefficient. That takes a strong signal. Feature B only becomes active when the penalty is already very small — meaning its signal is weak and only visible when almost no penalty is applied. In the real-estate analogy: Feature A is the last property you'd ever sell, Feature B is among the first.

</details>

---

**Question 2:** Why does the Ridge regularization path look smooth while the Lasso path has kinks (abrupt changes in slope)?

<details>
<summary>Show Answer</summary>

The difference comes from the **geometry of the constraint regions**:

- **Ridge (L2):** The penalty ball is a **sphere** — smooth everywhere, no corners. The loss function's elliptical contours touch the sphere at a smooth point, and as α changes the solution moves continuously along the sphere surface. No coordinate ever hits exactly zero.
- **Lasso (L1):** The penalty ball is a **diamond (hypercube in high dimensions)** — it has corners and edges. The loss contours are very likely to touch the diamond *at a corner*, which corresponds to exactly one (or more) coordinates being zero. As α changes, the active set can suddenly change (a feature jumps in or out), creating a kink in the path.

Mathematically, the Lasso solution path is **piecewise linear** (from the LARS result), while the Ridge solution path is a **smooth rational function** of α.

</details>

---

**Question 3:** A data scientist runs Lasso and 6 of 15 features survive at optimal α. Does this mean the other 9 features are useless? What might you be missing?

<details>
<summary>Show Answer</summary>

**Not necessarily.** Here are key things you might be missing:

1. **Correlated features:** Lasso arbitrarily picks one from a group of correlated features and zeros out the rest. The zeroed features are not useless — they are redundant *given* the chosen one. ElasticNet or grouped Lasso would handle this better.

2. **Non-linear relationships:** Lasso only captures linear associations. A feature with a quadratic or interaction effect might appear unimportant to Lasso but matter a great deal in a tree-based model.

3. **Wrong alpha:** The "optimal" α depends on the CV setup (number of folds, scoring metric). A slightly different CV could yield a different active set.

4. **Sample size:** With only 300 samples, some real but weak signals may not be detectable. With 10× the data, more features might survive.

5. **Path vs. final model:** Use the regularization path to *understand* feature importance order, but validate with domain knowledge and permutation importance for the final decision.

</details>